In [1]:
"""
patchhar_v2_profiler.py  (v2 — no dummy assumptions)
======================================================
Comprehensive inference profiling & interpretability analysis for PatchHAR v2
(Capture-24 / patchhar_v2.py edition).

WHAT WAS WRONG IN v1 AND IS NOW FIXED
───────────────────────────────────────
1.  DUMMY TIME VECTORS  (_dummy_times used torch.rand — uniform [0,1])
    ▶  All real timestamps in Capture-24 follow circadian structure.
       rand() produces flat, unbiased time vectors that give WRONG
       circadian-bias measurements and distort latency (different code path
       in circ_bias MLP than real data).
    FIX: latency benchmark now draws real timestamps from the dataset and
         reuses a fixed pool of them, preserving the true distribution.

2.  DUMMY SIGNAL STATISTICS  (_dummy_patches used torch.randn — N(0,1))
    ▶  Real windows are per-channel normalised and clipped to [-10, 10].
       randn occasionally produces values far outside this range, making
       FFT magnitudes (C1 DualDomain) unrealistically large and changing
       the code path through SiLU gates.
    FIX: real normalised windows from the dataset are used everywhere.

3.  GRADIENT ATTENTION — attn.grad WAS ALWAYS NONE
    ▶  The patched forward stored `attn` as a non-leaf intermediate tensor.
       Non-leaf tensors do not accumulate .grad by default.  The fallback
       was `torch.zeros_like(attn)`, silently producing a zero importance map.
    FIX: we register a tensor hook on `attn` with `attn.register_hook()`
         to capture its gradient during backward, which works for
         non-leaf tensors.

4.  MEMORY BENCHMARK USED PLAIN CE LOSS (not SmoothCE + all contributions)
    ▶  Training-step memory was measured with F.cross_entropy on a fresh
       untrained model, missing the SmoothCE + TC + recon losses that the
       real training loop uses.  This under-estimated peak memory.
    FIX: the training-step memory benchmark now replicates the actual
         loss function composition (SmoothCE + TC + recon) with real data.

5.  COLD-START BENCHMARK USED RANDOM WEIGHTS MODEL
    ▶  cold-start was measured on a freshly initialised PatchHARv2() —
       which is fine for timing but the time-vector must come from the
       real data pool.
    FIX: real data pool used for all timing.

6.  ATTENTION ROLLOUT USED ONLY ONE-LAYER ROLLOUT (no GDN layers)
    ▶  The formula A_rollout = 0.5*A + 0.5*I is correct for a single
       self-attention layer.  The docstring now correctly states this.
       GDN influence is captured separately via grad-attn.
    FIX: docstring corrected; rollout formula unchanged (correct for 1 layer).

7.  McNemar TEST ONLY VS MAJORITY CLASS
    ▶  Majority-class baseline is trivial; reviewers ask for more.
    FIX: McNemar now tests against BOTH majority-class AND random prediction.

8.  THROUGHPUT N_rep WAS TOO LOW FOR SMALL BATCHES
    ▶  max(1, 200 // B) gives only 200 total samples for B=1.
    FIX: minimum 500 total samples per batch size.

9.  PID FALLBACK COULD SILENTLY USE TRAIN SET FOR ALL METRICS
    ▶  If test_pids and val_pids were both empty the code fell back to
       train_pids[:5], producing optimistically inflated metrics.
    FIX: explicit RuntimeError raised instead of silent fallback.

10. FLOPS VIA THOP USED DUMMY PATCHES
    ▶  thop counts MACs using dummy randn() data.
    FIX: real normalised patches from the pool are used; note printed about
         FFT MACs being approximate in thop's tracer.
"""

from __future__ import annotations
import argparse, json, math, os, sys, time, tempfile, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path(__file__).parent))
from patchhar_v2 import (
    PatchHARv2, cfg, CC,
    WindowDataset, _collate,
    SmoothCE,
    train_pids, val_pids, test_pids,
    CLASSES, class_to_idx, idx_to_class, NUM_CLASSES,
    kappa as kappa_fn, mcc as mcc_fn,
    apply_rope,
    tc_loss, recon_loss,
    device, GPU,
)

print(f"Device : {device}")
if GPU:
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════════════════════════
# Real-data pool  (fixes #1, #2, #5, #8, #10)
# ════════════════════════════════════════════════════════════════════════════

class RealDataPool:
    """
    Caches real normalised windows + true timestamps from the dataset.
    Used for every benchmark so signal statistics and circadian time
    distributions match the actual inference workload exactly.
    """
    def __init__(self, pid_list: List[str], n_windows: int = 256):
        print(f"  Building real-data pool  ({n_windows} windows) ...")
        ds = WindowDataset(pid_list, cfg.PROC_DIR, class_to_idx, is_train=False)
        if len(ds) == 0:
            raise RuntimeError(
                f"No windows found for pids {pid_list}. "
                "Check cfg.PROC_DIR and that .npz files exist.")

        indices = list(range(min(n_windows, len(ds))))
        items   = [ds[i] for i in indices]
        patches_lists, times_l, labels_l, _, _, segs_l = zip(*items)

        n_scales = len(patches_lists[0])
        self.patches: List[torch.Tensor] = [
            torch.stack([b[s] for b in patches_lists]).to(device).float()
            for s in range(n_scales)
        ]
        self.times:  torch.Tensor = torch.stack(times_l).to(device).float()
        self.labels: torch.Tensor = torch.stack(labels_l).to(device)
        self.segs:   torch.Tensor = torch.stack(segs_l).to(device).float()
        self.n = len(items)
        print(f"  Pool: {self.n} windows | "
              f"time range [{self.times.min():.3f}, {self.times.max():.3f}] | "
              f"signal range [{self.patches[0].min():.2f}, "
              f"{self.patches[0].max():.2f}]")

    def batch(self, B: int, start: int = 0
              ) -> Tuple[List[torch.Tensor], torch.Tensor,
                         torch.Tensor, torch.Tensor]:
        idx     = [i % self.n for i in range(start, start + B)]
        patches = [p[idx] for p in self.patches]
        return patches, self.times[idx], self.labels[idx], self.segs[idx]


def _sync():
    if GPU:
        torch.cuda.synchronize()


def _make_loader(pid_list: List[str], batch_size: int = 64) -> DataLoader:
    ds = WindowDataset(pid_list, cfg.PROC_DIR, class_to_idx, is_train=False)
    if len(ds) == 0:
        raise RuntimeError(f"Empty dataset for pids: {pid_list}.")
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=0, pin_memory=GPU, collate_fn=_collate)


# ════════════════════════════════════════════════════════════════════════════
# 1.  MODEL COMPLEXITY
# ════════════════════════════════════════════════════════════════════════════

def report_complexity(model: PatchHARv2, pool: RealDataPool) -> Dict:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    buffers   = sum(p.numel() for p in model.buffers())

    module_params: Dict[str, int] = {
        name: sum(p.numel() for p in mod.parameters())
        for name, mod in model.named_children()
        if sum(p.numel() for p in mod.parameters()) > 0
    }

    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        fp32_path = f.name
    torch.save(model.state_dict(), fp32_path)
    size_fp32_mb = round(os.path.getsize(fp32_path) / 1e6, 2)
    os.unlink(fp32_path)

    fp16_sd = {k: v.half() if v.is_floating_point() else v
               for k, v in model.state_dict().items()}
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        fp16_path = f.name
    torch.save(fp16_sd, fp16_path)
    size_fp16_mb = round(os.path.getsize(fp16_path) / 1e6, 2)
    os.unlink(fp16_path)

    # FLOPs — use real data from pool (fix #10)
    flops_g = None
    try:
        from thop import profile as thop_profile
        p1, t1, _, _ = pool.batch(1)
        with torch.no_grad():
            macs, _ = thop_profile(model, inputs=(p1, t1), verbose=False)
        flops_g = round(2 * macs / 1e9, 4)
        print("  NOTE: FFT MACs in DualDomainPatchEmbed are approximate "
              "in thop's tracer.")
    except Exception as ex:
        print(f"  [FLOPs] thop unavailable: {ex}")

    result = dict(
        total_params=total, trainable_params=trainable,
        buffer_params=buffers, size_fp32_mb=size_fp32_mb,
        size_fp16_mb=size_fp16_mb, flops_gflops=flops_g,
        module_params=module_params,
        d_model=cfg.D_MODEL, n_heads=cfg.N_HEADS,
        n_layers=cfg.N_LAYERS, n_experts=cfg.N_EXPERTS,
        n_patches=cfg.N_PATCHES, patch_len=cfg.PATCH_LEN,
        window_size=cfg.WINDOW_SIZE, signal_rate=cfg.SIGNAL_RATE,
        channels=cfg.CHANNELS, num_classes=NUM_CLASSES,
    )

    print("\n── MODEL COMPLEXITY ─────────────────────────────────────────────")
    for k, v in result.items():
        if k != "module_params":
            print(f"  {k:<28}: {v}")
    print("  Per-module parameters:")
    for k, v in module_params.items():
        print(f"    {k:<26}: {v:,}")
    return result


# ════════════════════════════════════════════════════════════════════════════
# 2.  LATENCY & THROUGHPUT  (real data — fixes #1, #2, #8)
# ════════════════════════════════════════════════════════════════════════════

def benchmark_latency(model: PatchHARv2,
                      pool:  RealDataPool,
                      n_warmup: int = 50,
                      n_runs:   int = 500,
                      batch_sizes: Optional[List[int]] = None) -> Dict:
    if batch_sizes is None:
        batch_sizes = [1, 4, 8, 16, 32, 64]

    model.eval()
    results: Dict = {}

    for dev_str in (["cuda"] if GPU else []) + ["cpu"]:
        dev = torch.device(dev_str)
        m   = model.to(dev); m.eval()

        rp, rt, _, _ = pool.batch(1)
        rp = [p.to(dev) for p in rp]; rt = rt.to(dev)

        with torch.no_grad():
            for _ in range(n_warmup):
                m(rp, rt)
        if dev_str == "cuda":
            torch.cuda.synchronize()

        latencies = []
        with torch.no_grad():
            for i in range(n_runs):
                rp_i, rt_i, _, _ = pool.batch(1, start=i)
                rp_i = [p.to(dev) for p in rp_i]; rt_i = rt_i.to(dev)
                t0 = time.perf_counter()
                m(rp_i, rt_i)
                if dev_str == "cuda":
                    torch.cuda.synchronize()
                latencies.append((time.perf_counter() - t0) * 1e3)

        lat = np.array(latencies)
        results[f"latency_single_{dev_str}_ms"] = dict(
            mean=round(float(lat.mean()), 4), std=round(float(lat.std()), 4),
            p50=round(float(np.percentile(lat, 50)), 4),
            p95=round(float(np.percentile(lat, 95)), 4),
            p99=round(float(np.percentile(lat, 99)), 4),
            min=round(float(lat.min()), 4), max=round(float(lat.max()), 4),
        )
        print(f"\n  Latency B=1 ({dev_str.upper()}):  "
              f"mean={lat.mean():.3f} ms  p95={np.percentile(lat,95):.3f} ms  "
              f"p99={np.percentile(lat,99):.3f} ms")

    model.to(device)

    # Batch throughput — min 500 total samples (fix #8)
    throughput: Dict = {}
    with torch.no_grad():
        for B in batch_sizes:
            rp, rt, _, _ = pool.batch(B)
            for _ in range(20):
                model(rp, rt)
            _sync()
            N_rep = max(1, 500 // B)
            t0    = time.perf_counter()
            for i in range(N_rep):
                rp_i, rt_i, _, _ = pool.batch(B, start=i * B)
                model(rp_i, rt_i)
            _sync()
            elapsed = time.perf_counter() - t0
            sps = round(B * N_rep / elapsed, 1)
            throughput[f"B{B}"] = dict(
                samples_per_sec=sps,
                ms_per_sample=round(elapsed / (B * N_rep) * 1e3, 4))
            print(f"  Throughput B={B:3d}: {sps:9.1f} samp/s  "
                  f"({throughput[f'B{B}']['ms_per_sample']:.3f} ms/sample)")
    results["throughput"] = throughput

    # Cold-start (fix #5: real data)
    m_cold = PatchHARv2().to(device); m_cold.eval()
    rp, rt, _, _ = pool.batch(1)
    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        m_cold(rp, rt)
    _sync()
    results["cold_start_ms"] = round((time.perf_counter() - t0) * 1e3, 4)
    print(f"  Cold start: {results['cold_start_ms']:.2f} ms")

    # Warm-up curve
    m_wu = PatchHARv2().to(device); m_wu.eval()
    curve = []
    with torch.no_grad():
        for i in range(100):
            rp_i, rt_i, _, _ = pool.batch(1, start=i)
            t0 = time.perf_counter()
            m_wu(rp_i, rt_i)
            _sync()
            curve.append(round((time.perf_counter() - t0) * 1e3, 4))
    results["warmup_curve_ms"] = curve
    return results


# ════════════════════════════════════════════════════════════════════════════
# 3.  MEMORY  (fix #4: real loss composition)
# ════════════════════════════════════════════════════════════════════════════

def report_memory(model: PatchHARv2, pool: RealDataPool) -> Dict:
    results: Dict = {}
    if not GPU:
        print("  Memory profiling skipped (CPU-only run)")
        results["note"] = "CPU-only run"
        return results

    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    model.to(device).eval()
    rp, rt, _, _ = pool.batch(32)
    with torch.no_grad():
        model(rp, rt)
    torch.cuda.synchronize()
    results["peak_inference_b32_mb"] = round(
        torch.cuda.max_memory_allocated() / 1e6, 2)

    # Training step mirrors real training loop exactly (fix #4)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    m_tr = PatchHARv2().to(device); m_tr.train()
    opt  = torch.optim.AdamW(m_tr.parameters(), lr=cfg.LR,
                              weight_decay=cfg.WEIGHT_DECAY)
    cw   = torch.ones(NUM_CLASSES, device=device) / NUM_CLASSES
    crit = SmoothCE(weight=cw,
                    smoothing=cfg.LABEL_SMOOTH_EPS if CC.C6_LABEL_SMOOTH_TEMP
                    else 0.0)
    rp_tr, rt_tr, lbl_tr, seg_tr = pool.batch(32)

    if CC.C9_MANIFOLD_MIXUP:
        z = m_tr._embed_patches(rp_tr)
        z = z + (m_tr.circ_bias(rt_tr) if CC.C3_CIRCADIAN_BIAS
                 else m_tr.time_emb(rt_tr).unsqueeze(1))
        z = m_tr.input_norm(z)
        hiddens = []
        for layer in m_tr.delta_layers:
            z = layer(z); hiddens.append(z)
        if CC.C2_CALANET_SKIP_AGG:
            z = z + m_tr.skip_agg(hiddens)
        z = z + m_tr.moe1(z); z = m_tr.attn(z, m_tr.freqs); z = z + m_tr.moe2(z)
        z_pool = z.mean(1)
        logits = m_tr.head(z_pool)
        if CC.C6_LABEL_SMOOTH_TEMP:
            logits = logits / torch.exp(m_tr.log_tau).clamp(0.5, 2.0)
        loss = crit(logits, lbl_tr)
        if cfg.TC_LAMBDA > 0:
            loss = loss + cfg.TC_LAMBDA * tc_loss(logits)
        if CC.C10_RECON_AUX_GRAD_SURGERY:
            loss = loss + cfg.RECON_LAMBDA * recon_loss(
                m_tr.recon_head(z_pool.detach()), seg_tr)
    else:
        logits, recon = m_tr(rp_tr, rt_tr)
        loss = crit(logits, lbl_tr)
        if cfg.TC_LAMBDA > 0:
            loss = loss + cfg.TC_LAMBDA * tc_loss(logits)
        if CC.C10_RECON_AUX_GRAD_SURGERY and recon is not None:
            loss = loss + cfg.RECON_LAMBDA * recon_loss(recon, seg_tr)

    loss.backward(); opt.step(); torch.cuda.synchronize()
    results["peak_train_step_b32_mb"] = round(
        torch.cuda.max_memory_allocated() / 1e6, 2)
    del m_tr

    print(f"\n── MEMORY ───────────────────────────────────────────────────────")
    print(f"  Peak inference   (B=32): {results['peak_inference_b32_mb']:.1f} MB")
    print(f"  Peak train step  (B=32): {results['peak_train_step_b32_mb']:.1f} MB")
    return results


# ════════════════════════════════════════════════════════════════════════════
# 4.  ATTENTION ROLLOUT
# ════════════════════════════════════════════════════════════════════════════

def compute_attention_rollout(model: PatchHARv2,
                               loader: DataLoader,
                               max_samples: int = 500
                               ) -> Tuple[np.ndarray, np.ndarray]:
    """
    PatchHAR v2 has ONE GatedAttention layer.
    Rollout for a single layer:  R = 0.5*A + 0.5*I, row-normalised.
    Global importance = mean incoming attention per patch token.
    GDN-layer influence is captured separately via gradient attention.
    """
    model.eval()
    captured_attn: List[torch.Tensor] = []
    original_fwd  = model.attn.forward

    def _patched(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
        h = model.attn.norm(x); B, N, D = h.shape
        qkv = (model.attn.qkv(h)
               .reshape(B, N, 3, model.attn.h, model.attn.dh)
               .permute(0, 2, 1, 3, 4))
        q = qkv[:,0].transpose(1,2); k = qkv[:,1].transpose(1,2)
        v = qkv[:,2].transpose(1,2)
        q, k  = apply_rope(q, k, freqs)
        attn  = torch.softmax(
            (q @ k.transpose(-2,-1)) / math.sqrt(model.attn.dh), dim=-1)
        captured_attn.append(attn.detach().cpu())
        y = model.attn.out(
            (model.attn.drop(attn) @ v)
            .transpose(1,2).contiguous().reshape(B, N, D))
        return x + torch.sigmoid(model.attn.gate(h)) * y

    model.attn.forward = _patched
    all_rollout: List[np.ndarray] = []; all_labels: List[int] = []
    n_collected = 0

    with torch.no_grad():
        for batch in loader:
            if n_collected >= max_samples: break
            patches_list, times, labels, _, _, _ = batch
            captured_attn.clear()
            _ = model([p.to(device).float() for p in patches_list],
                      times.to(device).float())
            if not captured_attn: continue
            A = captured_attn[0].mean(dim=1).numpy()   # (B, NP, NP)
            I = np.eye(A.shape[-1])[None]
            A = 0.5*A + 0.5*I
            A = A / (A.sum(axis=-1, keepdims=True) + 1e-8)
            all_rollout.append(A.mean(axis=1))
            all_labels.extend(labels.numpy().tolist())
            n_collected += len(labels)

    model.attn.forward = original_fwd
    rollout = np.concatenate(all_rollout)[:max_samples].astype(np.float32)
    labels  = np.array(all_labels[:max_samples], dtype=np.int32)
    print(f"\n  Attention rollout: {rollout.shape}")
    return rollout, labels


# ════════════════════════════════════════════════════════════════════════════
# 5.  GRADIENT ATTENTION  (fix #3: register_hook for non-leaf grad)
# ════════════════════════════════════════════════════════════════════════════

def compute_grad_attention(model: PatchHARv2,
                            loader: DataLoader,
                            max_samples: int = 200) -> np.ndarray:
    """
    Uses attn.register_hook() instead of attn.grad so the gradient is
    captured correctly even though attn is a non-leaf intermediate tensor.
    """
    model.eval()
    original_fwd  = model.attn.forward
    _attn_buf:      List[torch.Tensor] = []
    _attn_grad_buf: List[torch.Tensor] = []
    all_grad_attn: List[np.ndarray]    = []
    n_collected = 0

    def _patched_grad(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
        h = model.attn.norm(x); B, N, D = h.shape
        qkv = (model.attn.qkv(h)
               .reshape(B, N, 3, model.attn.h, model.attn.dh)
               .permute(0, 2, 1, 3, 4))
        q = qkv[:,0].transpose(1,2); k = qkv[:,1].transpose(1,2)
        v = qkv[:,2].transpose(1,2)
        q, k  = apply_rope(q, k, freqs)
        attn  = torch.softmax(
            (q @ k.transpose(-2,-1)) / math.sqrt(model.attn.dh), dim=-1)
        _attn_buf.clear(); _attn_grad_buf.clear()
        _attn_buf.append(attn)
        # register_hook works on non-leaf tensors (fix #3)
        attn.register_hook(lambda g: _attn_grad_buf.append(g.detach()))
        y = model.attn.out(
            (model.attn.drop(attn) @ v)
            .transpose(1,2).contiguous().reshape(B, N, D))
        return x + torch.sigmoid(model.attn.gate(h)) * y

    model.attn.forward = _patched_grad

    for batch in loader:
        if n_collected >= max_samples: break
        patches_list, times, labels, _, _, _ = batch
        labels_d = labels.to(device)
        _attn_buf.clear(); _attn_grad_buf.clear()
        model.zero_grad()
        logits, _ = model([p.to(device).float() for p in patches_list],
                          times.to(device).float())
        if not _attn_buf: continue
        logits.gather(1, labels_d.unsqueeze(1)).sum().backward()
        if not _attn_grad_buf:
            n_collected += len(labels); continue
        with torch.no_grad():
            attn = _attn_buf[0].detach()
            grad = _attn_grad_buf[0]
            imp  = F.relu(grad * attn).mean(dim=1).sum(dim=-1)  # (B, NP)
            all_grad_attn.append(imp.cpu().numpy())
        n_collected += len(labels)
        model.zero_grad()

    model.attn.forward = original_fwd
    if not all_grad_attn:
        print("  [WARN] Grad-attn collection returned nothing.")
        return np.zeros((1, cfg.N_PATCHES), dtype=np.float32)

    ga = np.concatenate(all_grad_attn)[:max_samples]
    ga = (ga / (ga.max(axis=1, keepdims=True) + 1e-8)).astype(np.float32)
    print(f"  Gradient attention: {ga.shape}")
    return ga


# ════════════════════════════════════════════════════════════════════════════
# 6.  [C3] CIRCADIAN BIAS  (real timestamps only)
# ════════════════════════════════════════════════════════════════════════════

def analyse_circadian_bias(model: PatchHARv2,
                            loader: DataLoader,
                            max_samples: int = 500) -> Dict:
    if not CC.C3_CIRCADIAN_BIAS:
        print("  C3 disabled — skipping"); return {}
    model.eval()
    all_mag: List[np.ndarray] = []; all_labels: List[int] = []
    n_collected = 0
    with torch.no_grad():
        for batch in loader:
            if n_collected >= max_samples: break
            _, times, labels, _, _, _ = batch
            bias = model.circ_bias(times.to(device).float())  # (B, NP, D)
            all_mag.append(bias.abs().mean(dim=-1).cpu().numpy())
            all_labels.extend(labels.numpy().tolist())
            n_collected += len(labels)

    bias_arr   = np.concatenate(all_mag)[:max_samples]
    labels_arr = np.array(all_labels[:max_samples])
    mean_bias  = bias_arr.mean(axis=0)
    peak       = int(mean_bias.argmax())
    bias_per_class = {
        idx_to_class[i]: np.round(bias_arr[labels_arr==i].mean(axis=0), 5).tolist()
        for i in range(NUM_CLASSES) if (labels_arr==i).sum() > 0
    }
    np.save("circadian_bias_c24.npy", bias_arr)
    print(f"\n── C3 CIRCADIAN BIAS ────────────────────────────────────────────")
    print(f"  Shape: {bias_arr.shape}  range [{mean_bias.min():.4f}, "
          f"{mean_bias.max():.4f}]")
    print(f"  Peak patch: {peak}  "
          f"(≈{peak * cfg.PATCH_LEN / cfg.SIGNAL_RATE:.2f} s into window)")
    return dict(mean_bias_magnitude=np.round(mean_bias,6).tolist(),
                peak_patch=peak,
                peak_patch_time_s=round(peak*cfg.PATCH_LEN/cfg.SIGNAL_RATE,3),
                bias_per_class=bias_per_class)


# ════════════════════════════════════════════════════════════════════════════
# 7.  [C1] DUAL-DOMAIN GATES
# ════════════════════════════════════════════════════════════════════════════

def analyse_dual_domain_gates(model: PatchHARv2) -> Dict:
    if not CC.C1_DUAL_DOMAIN_EMBEDDING:
        print("  C1 disabled"); return {}
    pls = cfg.PATCH_LENS_MULTI if CC.C4_MULTISCALE_PATCHING else [cfg.PATCH_LEN]
    results: Dict = {}
    print("\n── C1 DUAL-DOMAIN GATES ─────────────────────────────────────────")
    for i, embed in enumerate(model.patch_embeds):
        with torch.no_grad():
            g = torch.sigmoid(embed.gate_w).cpu().numpy()
        key = f"PL{pls[i]}"
        results[key] = dict(
            mean_gate=round(float(g.mean()),6),
            std_gate=round(float(g.std()),6),
            frac_time_dom=round(float((g>0.5).mean()),4),
            frac_freq_dom=round(float((g<=0.5).mean()),4),
            gate_histogram=np.round(
                np.histogram(g,bins=10,range=(0,1))[0]/len(g),4).tolist())
        print(f"  Gate PL={pls[i]:<4}  mean={results[key]['mean_gate']:.4f}  "
              f"time>{results[key]['frac_time_dom']:.1%}  "
              f"freq>{results[key]['frac_freq_dom']:.1%}")
    return results


# ════════════════════════════════════════════════════════════════════════════
# 8.  [C2] SKIP-AGG WEIGHTS
# ════════════════════════════════════════════════════════════════════════════

def report_skip_weights(model: PatchHARv2) -> Dict:
    if not CC.C2_CALANET_SKIP_AGG: return {}
    with torch.no_grad():
        w = torch.softmax(model.skip_agg.weights, dim=0).cpu().numpy()
    print(f"\n── C2 SKIP-AGG WEIGHTS ──────────────────────────────────────────")
    for i, wi in enumerate(w):
        print(f"    Layer {i+1}: {wi:.4f}  {'█'*max(1,int(wi*40))}")
    return {"skip_agg_weights": np.round(w,6).tolist()}


# ════════════════════════════════════════════════════════════════════════════
# 9.  EXPERT ROUTING
# ════════════════════════════════════════════════════════════════════════════

def compute_expert_routing(model: PatchHARv2,
                            loader: DataLoader,
                            max_samples: int = 500) -> Dict:
    model.eval()
    routing1: List[torch.Tensor] = []; routing2: List[torch.Tensor] = []

    def _mk_hook(buf):
        def _h(module, inp, _out):
            with torch.no_grad():
                buf.append(torch.softmax(
                    module.router(inp[0]),dim=-1).detach().cpu())
        return _h

    h1 = model.moe1.register_forward_hook(_mk_hook(routing1))
    h2 = model.moe2.register_forward_hook(_mk_hook(routing2))
    n_collected = 0
    with torch.no_grad():
        for batch in loader:
            if n_collected >= max_samples: break
            patches_list, times, labels, _, _, _ = batch
            _ = model([p.to(device).float() for p in patches_list],
                      times.to(device).float())
            n_collected += len(labels)
    h1.remove(); h2.remove()

    def _analyse(buf, name):
        if not buf: return {}
        r    = torch.cat(buf).reshape(-1, buf[0].shape[-1]).numpy()
        load = r.mean(axis=0)
        ent  = float(-np.sum(load * np.log(load+1e-8)))
        idl  = math.log(len(load))
        print(f"  {name}: {np.round(load,3).tolist()}  "
              f"ent={ent:.4f}/{idl:.4f} ({ent/idl*100:.1f}% uniform)")
        return dict(load=np.round(load,6).tolist(), entropy=round(ent,6),
                    ideal_entropy=round(idl,6),
                    pct_of_uniform=round(ent/idl*100,2),
                    load_std=round(float(r.std(axis=0).mean()),6))

    print(f"\n── EXPERT ROUTING ───────────────────────────────────────────────")
    return dict(moe1=_analyse(routing1,"MoE1"), moe2=_analyse(routing2,"MoE2"))


# ════════════════════════════════════════════════════════════════════════════
# 10.  CALIBRATION
# ════════════════════════════════════════════════════════════════════════════

def compute_calibration(logits: torch.Tensor,
                         labels: np.ndarray,
                         n_bins: int = 15) -> Dict:
    probs   = torch.softmax(logits,dim=-1).numpy()
    confs   = probs.max(axis=1)
    correct = (probs.argmax(axis=1)==labels).astype(float)
    edges   = np.linspace(0,1,n_bins+1)
    bc      = np.zeros(n_bins); ba = np.zeros(n_bins); bn = np.zeros(n_bins)
    for i in range(n_bins):
        m = (confs>=edges[i])&(confs<edges[i+1])
        if m.sum()==0: continue
        bc[i]=confs[m].mean(); ba[i]=correct[m].mean(); bn[i]=m.sum()
    w   = bn/max(bn.sum(),1)
    ece = float(np.sum(w*np.abs(ba-bc)))
    mce = float(np.max(np.abs(ba-bc)))
    print(f"\n── CALIBRATION ──────────────────────────────────────────────────")
    print(f"  ECE = {ece:.5f}  |  MCE = {mce:.5f}")
    return dict(ECE=round(ece,6), MCE=round(mce,6),
                bin_conf=np.round(bc,5).tolist(),
                bin_acc=np.round(ba,5).tolist(),
                bin_count=bn.tolist(), n_bins=n_bins)


# ════════════════════════════════════════════════════════════════════════════
# 11.  McNemar — fix #7: two baselines
# ════════════════════════════════════════════════════════════════════════════

def mcnemar_test(pred_a: np.ndarray, pred_b: np.ndarray,
                 labels: np.ndarray,
                 label_a: str="model", label_b: str="baseline") -> Dict:
    from scipy.stats import chi2 as chi2_dist
    ok_a=(pred_a==labels); ok_b=(pred_b==labels)
    b=int(np.sum(ok_a&~ok_b)); c=int(np.sum(~ok_a&ok_b))
    if b+c==0:
        return dict(chi2=0.,p_value=1.,b=0,c=0,
                    label_a=label_a,label_b=label_b,
                    note="No discordant pairs.")
    chi2_val=float((abs(b-c)-1)**2/(b+c))
    p_val=float(1-chi2_dist.cdf(chi2_val,df=1))
    sig="significant (p<0.05)" if p_val<0.05 else "NOT significant"
    print(f"  McNemar [{label_a} vs {label_b}]:  "
          f"b={b} c={c}  χ²={chi2_val:.4f}  p={p_val:.4e}  → {sig}")
    return dict(chi2=round(chi2_val,6), p_value=round(p_val,8),
                b=b, c=c, label_a=label_a, label_b=label_b)


# ════════════════════════════════════════════════════════════════════════════
# 12.  Collect logits
# ════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def collect_logits(model: PatchHARv2,
                   loader: DataLoader) -> Tuple[torch.Tensor, np.ndarray]:
    model.eval()
    all_logits: List[torch.Tensor] = []; all_labels: List[int] = []
    for batch in loader:
        patches_list, times, labels, _, _, _ = batch
        logits, _ = model([p.to(device).float() for p in patches_list],
                          times.to(device).float())
        all_logits.append(logits.cpu()); all_labels.extend(labels.tolist())
    return torch.cat(all_logits), np.array(all_labels)


def per_class_metrics(pred: np.ndarray, labels: np.ndarray) -> Dict:
    from sklearn.metrics import precision_recall_fscore_support
    p,r,f,s = precision_recall_fscore_support(
        labels, pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    return {idx_to_class[i]: dict(precision=round(float(p[i]),4),
                                   recall=round(float(r[i]),4),
                                   f1=round(float(f[i]),4),
                                   support=int(s[i]))
            for i in range(NUM_CLASSES)}


# ════════════════════════════════════════════════════════════════════════════
# 13.  Paper table
# ════════════════════════════════════════════════════════════════════════════

def print_paper_table(results: Dict):
    W=68
    def _s(t): print(f"\n{'═'*W}\n  {t}\n{'─'*W}")
    def _r(k,v): print(f"  {k:<38}  {v}")

    print("\n"+"═"*W)
    print("  PATCHHAR v2  —  CAPTURE-24  —  PAPER METRICS SUMMARY")
    print("="*W)

    c=results.get("complexity",{})
    _s("ARCHITECTURE")
    _r("D / H / L / E",
       f"{c.get('d_model')} / {c.get('n_heads')} / "
       f"{c.get('n_layers')} / {c.get('n_experts')}")
    _r("Total parameters", f"{c.get('total_params',0):,}")
    _r("Model size FP32/FP16",
       f"{c.get('size_fp32_mb')} / {c.get('size_fp16_mb')} MB")
    if c.get("flops_gflops"):
        _r("FLOPs/sample", f"{c['flops_gflops']} GFLOPs")
    _r("Window/patches/PL/Hz",
       f"{c.get('window_size')}/{c.get('n_patches')}/"
       f"{c.get('patch_len')}/{c.get('signal_rate')}")

    lat=results.get("latency",{})
    for dev in ["cuda","cpu"]:
        key=f"latency_single_{dev}_ms"
        if key in lat:
            l=lat[key]; _s(f"LATENCY ({dev.upper()}, B=1, real data)")
            _r("Mean ± std",       f"{l['mean']} ± {l['std']} ms")
            _r("p50/p95/p99",      f"{l['p50']} / {l['p95']} / {l['p99']} ms")
            _r("Min/Max",          f"{l['min']} / {l['max']} ms")
    tp=lat.get("throughput",{})
    if tp:
        _s("THROUGHPUT  (≥500 samples/batch, real data)")
        for b,v in tp.items():
            _r(f"Batch {b}",
               f"{v['samples_per_sec']:>9.1f} samp/s  "
               f"({v['ms_per_sample']:.3f} ms/sample)")
    if lat.get("cold_start_ms"):
        _r("Cold start", f"{lat['cold_start_ms']} ms")

    mem=results.get("memory",{})
    if "peak_inference_b32_mb" in mem:
        _s("GPU MEMORY  (B=32, real loss)")
        _r("Peak inference",  f"{mem['peak_inference_b32_mb']} MB")
        _r("Peak train step", f"{mem['peak_train_step_b32_mb']} MB")

    cal=results.get("calibration",{})
    if cal:
        _s("CALIBRATION"); _r("ECE",cal.get("ECE")); _r("MCE",cal.get("MCE"))

    gm=results.get("global_metrics",{})
    if gm:
        _s("CLASSIFICATION PERFORMANCE  (test set)")
        for k,v in gm.items(): _r(k,v)

    pc=results.get("per_class_metrics",{})
    if pc:
        _s("PER-CLASS METRICS")
        print(f"  {'Class':<28} {'Prec':>7} {'Rec':>7} {'F1':>7} {'N':>7}")
        print("  "+"─"*54)
        for cls,m in pc.items():
            print(f"  {cls:<28} {m['precision']:>7.4f} "
                  f"{m['recall']:>7.4f} {m['f1']:>7.4f} {m['support']:>7}")

    er=results.get("expert_routing",{})
    if er:
        _s("EXPERT ROUTING")
        for moe,v in er.items():
            if v:
                _r(f"{moe} load",str([round(x,3) for x in v["load"]]))
                _r(f"{moe} entropy",
                   f"{v['entropy']:.4f}/{v['ideal_entropy']:.4f} "
                   f"({v['pct_of_uniform']:.1f}% uniform)")

    dd=results.get("dual_domain_gates",{})
    if dd:
        _s("C1 DUAL-DOMAIN GATE")
        for k,v in dd.items():
            _r(f"{k} mean gate",v["mean_gate"])
            _r(f"{k} time%",f"{v['frac_time_dom']:.1%}")
            _r(f"{k} freq%",f"{v['frac_freq_dom']:.1%}")

    cb=results.get("circadian_bias",{})
    if cb:
        _s("C3 CIRCADIAN BIAS  (real timestamps)")
        _r("Peak patch index",   cb.get("peak_patch"))
        _r("Peak patch time (s)",cb.get("peak_patch_time_s"))

    sw=results.get("skip_agg",{})
    if sw and "skip_agg_weights" in sw:
        _s("C2 SKIP-AGG LAYER WEIGHTS")
        for i,wi in enumerate(sw["skip_agg_weights"]):
            print(f"    Layer {i+1}: {wi:.4f}  {'▪'*max(1,int(wi*40))}")

    for mc_key, mc_lbl in [("mcnemar_majority","model vs majority-class"),
                             ("mcnemar_random",  "model vs random-prediction")]:
        mc=results.get(mc_key,{})
        if mc:
            _s(f"MCNEMAR TEST  ({mc_lbl})")
            _r("χ²",mc.get("chi2")); _r("p-value",mc.get("p_value"))
            _r("Discordant (b,c)",f"{mc.get('b')}, {mc.get('c')}")

    print("\n"+"═"*W+"\n")


# ════════════════════════════════════════════════════════════════════════════
# 14.  MAIN
# ════════════════════════════════════════════════════════════════════════════

def run_profiler(checkpoint_path: Optional[str]=None,
                 benchmark_only:  bool=False,
                 max_samples:     int=500,
                 n_latency_runs:  int=500):

    # fix #9: no silent train-set fallback
    test_pids_use = test_pids if test_pids else val_pids
    if not test_pids_use:
        raise RuntimeError(
            "No test or val participants found. "
            "Check TRAIN_N / VAL_N in patchhar_v2.py or add more .npz files.")

    pool  = RealDataPool(test_pids_use, n_windows=256)
    model = PatchHARv2().to(device); model.eval()

    if checkpoint_path and Path(checkpoint_path).exists():
        ckpt = torch.load(checkpoint_path, map_location=device,
                          weights_only=False)
        if isinstance(ckpt, dict) and "model" in ckpt:
            ckpt = ckpt["model"]
        model.load_state_dict(ckpt, strict=True)
        print(f"  Checkpoint loaded: {checkpoint_path}")
    else:
        print("  NOTE: No valid checkpoint — random weights.\n"
              "        Latency/complexity/memory valid; "
              "interpretability/classification are NOT.")

    all_results: Dict = {}
    all_results["complexity"] = report_complexity(model, pool)

    print("\n── LATENCY & THROUGHPUT ─────────────────────────────────────────")
    all_results["latency"] = benchmark_latency(
        model, pool, n_runs=n_latency_runs)
    all_results["memory"]  = report_memory(model, pool)

    if benchmark_only:
        print("\n  [benchmark_only] Skipping data-dependent analyses.")
        with open("paper_metrics_c24.json","w") as f:
            json.dump(all_results, f, indent=2)
        print_paper_table(all_results); return all_results

    loader = _make_loader(test_pids_use, batch_size=64)
    print(f"  {len(loader.dataset):,} test windows")

    print("\n── ATTENTION ROLLOUT ────────────────────────────────────────────")
    rollout, roll_labels = compute_attention_rollout(
        model, loader, max_samples)
    np.save("attn_rollout_c24.npy",   rollout)
    np.save("rollout_labels_c24.npy", roll_labels)
    all_results["attention_rollout_mean"] = np.round(
        rollout.mean(axis=0),5).tolist()
    all_results["attention_rollout_per_class"] = {
        idx_to_class[i]: np.round(rollout[roll_labels==i].mean(axis=0),5).tolist()
        for i in range(NUM_CLASSES) if (roll_labels==i).sum()>0
    }

    print("\n── GRADIENT ATTENTION ───────────────────────────────────────────")
    grad_attn = compute_grad_attention(model,loader,min(200,max_samples))
    np.save("grad_attn_c24.npy", grad_attn)
    all_results["grad_attention_mean"] = np.round(
        grad_attn.mean(axis=0),5).tolist()

    all_results["circadian_bias"]   = analyse_circadian_bias(
        model,loader,max_samples)
    all_results["dual_domain_gates"] = analyse_dual_domain_gates(model)
    all_results["skip_agg"]         = report_skip_weights(model)
    all_results["expert_routing"]   = compute_expert_routing(
        model,loader,max_samples)

    print("\n  Collecting logits ...")
    logits_all, labels_all = collect_logits(model, loader)

    calib = compute_calibration(logits_all, labels_all)
    all_results["calibration"] = calib
    np.save("calibration_c24.npy",
            np.stack([calib["bin_conf"],calib["bin_acc"],calib["bin_count"]]))

    from sklearn.metrics import f1_score as sk_f1
    pred_all = logits_all.argmax(1).numpy()
    all_results["global_metrics"] = dict(
        macro_f1   =round(float(sk_f1(labels_all,pred_all,average="macro",   zero_division=0)),4),
        micro_f1   =round(float(sk_f1(labels_all,pred_all,average="micro",   zero_division=0)),4),
        weighted_f1=round(float(sk_f1(labels_all,pred_all,average="weighted",zero_division=0)),4),
        accuracy   =round(float((pred_all==labels_all).mean()),4),
        cohen_kappa=round(float(kappa_fn(labels_all,pred_all)),4),
        mcc        =round(float(mcc_fn(labels_all,pred_all)),4),
    )
    print(f"\n── GLOBAL METRICS ───────────────────────────────────────────────")
    for k,v in all_results["global_metrics"].items():
        print(f"  {k:<28}: {v}")

    all_results["per_class_metrics"] = per_class_metrics(pred_all,labels_all)

    # Two baselines (fix #7)
    majority_pred = np.full_like(labels_all,int(np.bincount(labels_all).argmax()))
    random_pred   = np.random.randint(0, NUM_CLASSES, size=len(labels_all))
    print(f"\n── STATISTICAL TESTS ────────────────────────────────────────────")
    all_results["mcnemar_majority"] = mcnemar_test(
        pred_all,majority_pred,labels_all,
        label_a="PatchHAR-v2",label_b="majority-class")
    all_results["mcnemar_random"]   = mcnemar_test(
        pred_all,random_pred,labels_all,
        label_a="PatchHAR-v2",label_b="random-prediction")

    with open("paper_metrics_c24.json","w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n{'═'*68}")
    print("  Saved: paper_metrics_c24.json")
    print(f"         attn_rollout_c24.npy    ({rollout.shape})")
    print(f"         grad_attn_c24.npy       ({grad_attn.shape})")
    if CC.C3_CIRCADIAN_BIAS:
        print(f"         circadian_bias_c24.npy  (N, {cfg.N_PATCHES})")
    print(f"         calibration_c24.npy     (3, {calib['n_bins']})")
    print("═"*68)

    print_paper_table(all_results)
    return all_results


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint",     type=str, default=None)
    p.add_argument("--benchmark_only", action="store_true")
    p.add_argument("--max_samples",    type=int, default=500)
    p.add_argument("--n_latency_runs", type=int, default=500)
    args = p.parse_args()
    run_profiler(checkpoint_path=args.checkpoint,
                 benchmark_only=args.benchmark_only,
                 max_samples=args.max_samples,
                 n_latency_runs=args.n_latency_runs)

NameError: name '__file__' is not defined